# 🗺️ CrashLens — OSM Road Cache Generator (All 50 States + DC)

Downloads USA road network → **crash_enricher-compatible parquet** → gzip → R2.

| Feature | Detail |
|---------|--------|
| **Format** | Flat parquet with `mid_lat`, `mid_lon`, `node_id` — NOT GeoDataFrame |
| **States** | 50 states + DC = 51 total |
| **Gzipped** | ~40-60% smaller before upload |
| **Auto-skip** | Checks R2 first — resumes if Colab disconnects |
| **List-safe** | OSM list columns → semicolon strings (no ArrowTypeError) |
| **Est. time** | ~4-5 hours for all 51 |

⚠️ **v2 FIX:** Previous version saved GeoPandas format (geometry column).
crash_enricher.py needs flat format with `mid_lat`, `mid_lon`, `u_node`, `v_node`.
This version generates the correct format.

**Run Cells 1→4 in order. Cell 4 is fully automatic.**

## Cell 1 — Install (~60s)

In [ ]:
!pip install -q osmnx geopandas pandas pyarrow scipy boto3

import osmnx as ox
import geopandas as gpd
import pandas as pd
import boto3
import os, time, json, gc, gzip, shutil, math
from pathlib import Path
from collections import defaultdict
from IPython.display import display, HTML, clear_output

print(f'osmnx {ox.__version__}, geopandas {gpd.__version__}')
print('✅ Ready')

## Cell 2 — R2 Credentials

Same as GitHub Secrets. **Clear after use.**

In [ ]:
R2_ENDPOINT      = ''   # https://YOUR_ACCOUNT.r2.cloudflarestorage.com
R2_ACCESS_KEY_ID = ''   # R2 access key ID
R2_SECRET_KEY    = ''   # R2 secret access key
R2_BUCKET        = 'crash-lens-data'

assert R2_ENDPOINT and R2_ACCESS_KEY_ID and R2_SECRET_KEY, '❌ Fill in credentials'
s3 = boto3.client('s3', endpoint_url=R2_ENDPOINT,
    aws_access_key_id=R2_ACCESS_KEY_ID, aws_secret_access_key=R2_SECRET_KEY, region_name='auto')
try:
    s3.list_objects_v2(Bucket=R2_BUCKET, Prefix='delaware/', MaxKeys=1)
    print(f'✅ R2 connected — bucket: {R2_BUCKET}')
except Exception as e:
    print(f'❌ {e}')

## Cell 3 — State Registry (50 states + DC)

In [ ]:
ALL_STATES = [
    {"name": "Alabama", "abbreviation": "al", "r2_prefix": "alabama"},
    {"name": "Alaska", "abbreviation": "ak", "r2_prefix": "alaska"},
    {"name": "Arizona", "abbreviation": "az", "r2_prefix": "arizona"},
    {"name": "Arkansas", "abbreviation": "ar", "r2_prefix": "arkansas"},
    {"name": "California", "abbreviation": "ca", "r2_prefix": "california"},
    {"name": "Colorado", "abbreviation": "co", "r2_prefix": "colorado"},
    {"name": "Connecticut", "abbreviation": "ct", "r2_prefix": "connecticut"},
    {"name": "Delaware", "abbreviation": "de", "r2_prefix": "delaware"},
    {"name": "District of Columbia", "abbreviation": "dc", "r2_prefix": "district_of_columbia"},
    {"name": "Florida", "abbreviation": "fl", "r2_prefix": "florida"},
    {"name": "Georgia", "abbreviation": "ga", "r2_prefix": "georgia"},
    {"name": "Hawaii", "abbreviation": "hi", "r2_prefix": "hawaii"},
    {"name": "Idaho", "abbreviation": "id", "r2_prefix": "idaho"},
    {"name": "Illinois", "abbreviation": "il", "r2_prefix": "illinois"},
    {"name": "Indiana", "abbreviation": "in", "r2_prefix": "indiana"},
    {"name": "Iowa", "abbreviation": "ia", "r2_prefix": "iowa"},
    {"name": "Kansas", "abbreviation": "ks", "r2_prefix": "kansas"},
    {"name": "Kentucky", "abbreviation": "ky", "r2_prefix": "kentucky"},
    {"name": "Louisiana", "abbreviation": "la", "r2_prefix": "louisiana"},
    {"name": "Maine", "abbreviation": "me", "r2_prefix": "maine"},
    {"name": "Maryland", "abbreviation": "md", "r2_prefix": "maryland"},
    {"name": "Massachusetts", "abbreviation": "ma", "r2_prefix": "massachusetts"},
    {"name": "Michigan", "abbreviation": "mi", "r2_prefix": "michigan"},
    {"name": "Minnesota", "abbreviation": "mn", "r2_prefix": "minnesota"},
    {"name": "Mississippi", "abbreviation": "ms", "r2_prefix": "mississippi"},
    {"name": "Missouri", "abbreviation": "mo", "r2_prefix": "missouri"},
    {"name": "Montana", "abbreviation": "mt", "r2_prefix": "montana"},
    {"name": "Nebraska", "abbreviation": "ne", "r2_prefix": "nebraska"},
    {"name": "Nevada", "abbreviation": "nv", "r2_prefix": "nevada"},
    {"name": "New Hampshire", "abbreviation": "nh", "r2_prefix": "new_hampshire"},
    {"name": "New Jersey", "abbreviation": "nj", "r2_prefix": "new_jersey"},
    {"name": "New Mexico", "abbreviation": "nm", "r2_prefix": "new_mexico"},
    {"name": "New York", "abbreviation": "ny", "r2_prefix": "new_york"},
    {"name": "North Carolina", "abbreviation": "nc", "r2_prefix": "north_carolina"},
    {"name": "North Dakota", "abbreviation": "nd", "r2_prefix": "north_dakota"},
    {"name": "Ohio", "abbreviation": "oh", "r2_prefix": "ohio"},
    {"name": "Oklahoma", "abbreviation": "ok", "r2_prefix": "oklahoma"},
    {"name": "Oregon", "abbreviation": "or", "r2_prefix": "oregon"},
    {"name": "Pennsylvania", "abbreviation": "pa", "r2_prefix": "pennsylvania"},
    {"name": "Rhode Island", "abbreviation": "ri", "r2_prefix": "rhode_island"},
    {"name": "South Carolina", "abbreviation": "sc", "r2_prefix": "south_carolina"},
    {"name": "South Dakota", "abbreviation": "sd", "r2_prefix": "south_dakota"},
    {"name": "Tennessee", "abbreviation": "tn", "r2_prefix": "tennessee"},
    {"name": "Texas", "abbreviation": "tx", "r2_prefix": "texas"},
    {"name": "Utah", "abbreviation": "ut", "r2_prefix": "utah"},
    {"name": "Vermont", "abbreviation": "vt", "r2_prefix": "vermont"},
    {"name": "Virginia", "abbreviation": "va", "r2_prefix": "virginia"},
    {"name": "Washington", "abbreviation": "wa", "r2_prefix": "washington"},
    {"name": "West Virginia", "abbreviation": "wv", "r2_prefix": "west_virginia"},
    {"name": "Wisconsin", "abbreviation": "wi", "r2_prefix": "wisconsin"},
    {"name": "Wyoming", "abbreviation": "wy", "r2_prefix": "wyoming"},
]

print(f'{len(ALL_STATES)} states registered')

# ── FILTER (uncomment ONE) ──
# STATES_TO_RUN = [s for s in ALL_STATES if s['abbreviation'] == 'de']                    # Just Delaware
# STATES_TO_RUN = [s for s in ALL_STATES if s['abbreviation'] in ('de','va','co','md')]   # Specific
STATES_TO_RUN = ALL_STATES                                                                 # All 51

print(f'Will process: {len(STATES_TO_RUN)}')

## Cell 4 — Run Pipeline (crash_enricher-compatible format)

Generates **flat parquet** files matching what `crash_enricher.py` expects:
- Roads: `u_node, v_node, mid_lat, mid_lon, highway, name, ref, lanes, maxspeed, length_m`
- Intersections: `node_id, lat, lon, degree`

**NOT** GeoPandas format (no geometry column). This is the critical difference from v1.

In [ ]:
CACHE_DIR = Path('cache')
CACHE_DIR.mkdir(exist_ok=True)
results = {'completed': [], 'skipped': [], 'failed': []}
t_start = time.time()


def r2_exists(prefix, abbr):
    try:
        s3.head_object(Bucket=R2_BUCKET, Key=f'{prefix}/cache/{abbr}_roads.parquet.gz')
        return True
    except: return False


def r2_upload(local_path, r2_key):
    for attempt in range(3):
        try: s3.upload_file(str(local_path), R2_BUCKET, r2_key); return True
        except: time.sleep(2 ** (attempt + 1))
    return False


def gzip_file(src, dst):
    with open(src, 'rb') as fi, gzip.open(dst, 'wb', compresslevel=6) as fo:
        shutil.copyfileobj(fi, fo)
    raw = os.path.getsize(src) / 1048576
    gz = os.path.getsize(dst) / 1048576
    os.remove(src)
    return raw, gz


def show_progress(idx, name, msg):
    clear_output(wait=True)
    total = len(STATES_TO_RUN)
    d, sk, fl = len(results['completed']), len(results['skipped']), len(results['failed'])
    done = d + sk + fl
    pct = done / total * 100 if total else 0
    elapsed = time.time() - t_start
    eta = f'{(elapsed / max(done,1)) * (total-done) / 60:.0f} min' if done else '...'
    bar = '█' * int(40 * pct / 100) + '░' * (40 - int(40 * pct / 100))
    total_gz = sum(s.get('gz_mb', 0) for s in results['completed'])
    html = f'''<div style="font-family:monospace;padding:20px;background:#1e1e2e;color:#cdd6f4;border-radius:12px;max-width:700px;">
    <h3 style="color:#89b4fa;margin:0 0 12px;">🗺️ OSM Cache Generator v2 (crash_enricher format)</h3>
    <div style="font-size:20px;">{bar} {pct:.0f}%</div>
    <table style="color:#cdd6f4;font-size:14px;margin:12px 0;border-spacing:4px 6px;">
      <tr><td style="padding-right:20px;color:#a6adc8;">📍 Current:</td><td><b>{name}</b> — {msg}</td></tr>
      <tr><td style="color:#a6adc8;">✅ Done:</td><td><b>{d}</b> / {total}</td></tr>
      <tr><td style="color:#a6adc8;">⏭️ Skipped:</td><td>{sk}</td></tr>
      <tr><td style="color:#a6adc8;">❌ Failed:</td><td>{fl}</td></tr>
      <tr><td style="color:#a6adc8;">⏱️ Elapsed:</td><td>{elapsed/60:.1f} min</td></tr>
      <tr><td style="color:#a6adc8;">⏳ ETA:</td><td>~{eta}</td></tr>
      <tr><td style="color:#a6adc8;">💾 Uploaded:</td><td>{total_gz:.0f} MB</td></tr>
    </table>'''
    if results['completed']:
        last5 = results['completed'][-5:]
        html += '<div style="font-size:11px;color:#a6adc8;margin-top:4px;">Recent: '
        html += ' → '.join(f"{s['abbreviation'].upper()} ({s['gz_mb']:.0f}MB {s['sec']:.0f}s)" for s in last5)
        html += '</div>'
    if results['failed']:
        html += f'<div style="color:#f38ba8;font-size:11px;margin-top:6px;">Failed: {", ".join(s["name"] for s in results["failed"])}</div>'
    html += '</div>'
    display(HTML(html))


def convert_to_enricher_format(G, abbr):
    """
    Convert osmnx graph to crash_enricher-compatible flat DataFrames.

    crash_enricher.py expects:
      roads: u_node, v_node, u_lat, u_lon, v_lat, v_lon, mid_lat, mid_lon,
             highway, name, ref, oneway, lanes, maxspeed, length_m, bridge, tunnel
      intersections: node_id, lat, lon, degree

    NOT GeoPandas GeoDataFrames with geometry columns.
    """
    nodes_gdf, edges_gdf = ox.graph_to_gdfs(G, nodes=True, edges=True)

    # ── Roads (flat format) ──
    road_data = []
    for idx, row in edges_gdf.iterrows():
        u, v = idx[0], idx[1]
        coords = list(row.geometry.coords)
        u_lat, u_lon = coords[0][1], coords[0][0]
        v_lat, v_lon = coords[-1][1], coords[-1][0]

        # Clean list values → semicolon strings
        def _clean(val):
            if isinstance(val, (list, tuple)):
                return '; '.join(str(x) for x in val)
            return str(val) if val is not None and str(val) != 'nan' else ''

        road_data.append({
            'u_node': u, 'v_node': v,
            'u_lat': u_lat, 'u_lon': u_lon,
            'v_lat': v_lat, 'v_lon': v_lon,
            'mid_lat': (u_lat + v_lat) / 2,
            'mid_lon': (u_lon + v_lon) / 2,
            'highway':  _clean(row.get('highway', '')),
            'name':     _clean(row.get('name', '')),
            'ref':      _clean(row.get('ref', '')),
            'oneway':   _clean(row.get('oneway', '')),
            'lanes':    _clean(row.get('lanes', '')),
            'maxspeed': _clean(row.get('maxspeed', '')),
            'length_m': float(row.get('length', 0) or 0),
            'bridge':   _clean(row.get('bridge', '')),
            'tunnel':   _clean(row.get('tunnel', '')),
        })

    road_df = pd.DataFrame(road_data)

    # ── Intersections (flat format) ──
    degrees = dict(G.degree())
    int_data = []
    for node_id, deg in degrees.items():
        if deg >= 3:
            n = nodes_gdf.loc[node_id]
            int_data.append({
                'node_id': node_id,
                'lat': n.geometry.y,
                'lon': n.geometry.x,
                'degree': deg,
            })

    int_df = pd.DataFrame(int_data) if int_data else pd.DataFrame(columns=['node_id','lat','lon','degree'])

    return road_df, int_df


def process_state(state_info, idx):
    name = state_info['name']
    abbr = state_info['abbreviation']
    prefix = state_info['r2_prefix']

    show_progress(idx, name, 'Checking R2...')
    if r2_exists(prefix, abbr):
        results['skipped'].append({'name': name, 'abbreviation': abbr})
        return

    show_progress(idx, name, 'Downloading from OSM...')
    t0 = time.time()

    if name == 'District of Columbia': place = 'Washington, DC, United States'
    elif name == 'Georgia': place = 'State of Georgia, United States'
    elif name == 'Washington': place = 'State of Washington, United States'
    else: place = f'{name}, United States'

    try:
        G = ox.graph_from_place(place, network_type='drive', simplify=True)
    except:
        try:
            show_progress(idx, name, f'Retrying...')
            G = ox.graph_from_place(f'State of {name}, United States', network_type='drive', simplify=True)
        except Exception as e:
            results['failed'].append({'name': name, 'abbreviation': abbr, 'error': str(e)[:80]})
            return

    show_progress(idx, name, f'{G.number_of_nodes():,} nodes → converting to enricher format...')

    # Convert to crash_enricher-compatible flat format
    road_df, int_df = convert_to_enricher_format(G, abbr)

    # Save roads
    roads_pq = str(CACHE_DIR / f'{abbr}_roads.parquet')
    roads_gz = roads_pq + '.gz'
    road_df.to_parquet(roads_pq, index=False)
    raw_r, gz_r = gzip_file(roads_pq, roads_gz)

    # Save intersections
    int_pq = str(CACHE_DIR / f'{abbr}_intersections.parquet')
    int_gz = int_pq + '.gz'
    int_df.to_parquet(int_pq, index=False)
    raw_i, gz_i = gzip_file(int_pq, int_gz)

    sec = time.time() - t0
    show_progress(idx, name, f'Uploading {gz_r:.0f}+{gz_i:.0f} MB to R2...')

    ok1 = r2_upload(roads_gz, f'{prefix}/cache/{abbr}_roads.parquet.gz')
    ok2 = r2_upload(int_gz, f'{prefix}/cache/{abbr}_intersections.parquet.gz')

    Path(roads_gz).unlink(missing_ok=True)
    Path(int_gz).unlink(missing_ok=True)
    del G, road_df, int_df; gc.collect()

    if ok1 and ok2:
        results['completed'].append({
            'name': name, 'abbreviation': abbr,
            'gz_mb': gz_r + gz_i, 'raw_mb': raw_r + raw_i,
            'saved': (raw_r + raw_i) - (gz_r + gz_i), 'sec': sec,
        })
    else:
        results['failed'].append({'name': name, 'abbreviation': abbr, 'error': 'R2 upload failed'})


# ═══ RUN ═══
for i, state in enumerate(STATES_TO_RUN):
    try: process_state(state, i)
    except Exception as e:
        results['failed'].append({'name': state['name'], 'abbreviation': state['abbreviation'], 'error': str(e)[:80]})
    time.sleep(2)

show_progress(len(STATES_TO_RUN)-1, 'DONE', 'All states processed ✅')
el = time.time() - t_start
print(f'\n{"═"*60}')
print(f'  COMPLETE in {el/60:.1f} min')
print(f'  Downloaded: {len(results["completed"])}')
print(f'  Skipped:    {len(results["skipped"])}')
print(f'  Failed:     {len(results["failed"])}')
if results['completed']:
    total_gz = sum(s['gz_mb'] for s in results['completed'])
    total_raw = sum(s['raw_mb'] for s in results['completed'])
    print(f'  R2 storage: {total_gz:.0f} MB ({total_gz/1024:.1f} GB)')
    print(f'  Gzip saved: {(total_raw-total_gz):.0f} MB')
if results['failed']:
    print(f'  Failed: {", ".join(s["name"] for s in results["failed"])}')
print(f'{"═"*60}')

## Cell 5 — Verify R2 (optional)

In [ ]:
total_mb = 0; count = 0
for st in ALL_STATES:
    p, a = st['r2_prefix'], st['abbreviation']
    try:
        r = s3.list_objects_v2(Bucket=R2_BUCKET, Prefix=f'{p}/cache/{a}_')
        files = r.get('Contents', [])
        if files:
            mb = sum(f['Size'] for f in files) / 1048576
            total_mb += mb; count += 1
            fl = ', '.join(f"{os.path.basename(f['Key'])} ({f['Size']/1048576:.1f}MB)" for f in files)
            print(f'  ✅ {st["name"]:22s} ({a})  {fl}')
        else:
            print(f'  ⬜ {st["name"]:22s} ({a})')
    except: print(f'  ❌ {st["name"]:22s} ({a})')
print(f'\nCached: {count}/{len(ALL_STATES)} — {total_mb:.0f} MB ({total_mb/1024:.1f} GB)')

## Cell 6 — Retry Failed (optional)

In [ ]:
if results.get('failed'):
    retry = [s for s in ALL_STATES if s['abbreviation'] in {f['abbreviation'] for f in results['failed']}]
    print(f'Retrying {len(retry)}: {", ".join(s["name"] for s in retry)}')
    results['failed'] = []
    for i, st in enumerate(retry):
        try: process_state(st, i)
        except Exception as e:
            results['failed'].append({'name':st['name'],'abbreviation':st['abbreviation'],'error':str(e)[:80]})
        time.sleep(5)
    print(f'Done: {len(results["completed"])} ok, {len(results["failed"])} still failing')
else: print('No failures ✅')

## Cell 7 — Validate Parquet Format (optional)

Downloads one parquet from R2 and verifies it has the correct columns for crash_enricher.

In [ ]:
# Quick format validation — downloads Delaware cache and checks columns
import tempfile, io

EXPECTED_ROAD_COLS = {'u_node','v_node','mid_lat','mid_lon','highway','name','ref','length_m'}
EXPECTED_INT_COLS = {'node_id','lat','lon','degree'}

test_state = 'de'
test_prefix = 'delaware'

for file_type, expected in [('roads', EXPECTED_ROAD_COLS), ('intersections', EXPECTED_INT_COLS)]:
    key = f'{test_prefix}/cache/{test_state}_{file_type}.parquet.gz'
    try:
        obj = s3.get_object(Bucket=R2_BUCKET, Key=key)
        gz_bytes = obj['Body'].read()
        raw_bytes = gzip.decompress(gz_bytes)
        df = pd.read_parquet(io.BytesIO(raw_bytes))
        cols = set(df.columns)
        missing = expected - cols
        if missing:
            print(f'  ❌ {file_type}: WRONG FORMAT — missing {missing}')
            print(f'     Has: {list(df.columns)}')
            print(f'     This is probably GeoPandas format. Re-run Cell 4 for this state.')
        else:
            print(f'  ✅ {file_type}: correct crash_enricher format ({len(df):,} rows)')
            print(f'     Columns: {list(df.columns)}')
    except Exception as e:
        print(f'  ⚠️ {file_type}: {e}')